# Setup & Configuration

This notebook walks through the **configuration layer** of Equity Lake, including:

- Loading and inspecting `Settings` (Pydantic Settings with YAML + env vars)
- Exploring the **ticker universe** from `tickers.yaml`
- Understanding config priority: init > env vars > `.env` > YAML
- Visualizing ticker distribution by market, sector, and priority

**Prerequisites:** Run `uv sync` and ensure `config/settings.yaml` and `config/tickers.yaml` exist.

In [1]:
import sys
sys.path.insert(0, "../../src")

from equity_lake.core.config import get_settings, Settings
from equity_lake.core.config import TickerConfig
from equity_lake.core.config import get_settings
from equity_lake.core.paths import CONFIG_DIR
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path

print("Config directory:", CONFIG_DIR)
print("Settings YAML:", CONFIG_DIR / "settings.yaml")
print("Tickers YAML:", CONFIG_DIR / "tickers.yaml")

Config directory: /Users/minghao/Desktop/personal/equity_lake/config
Settings YAML: /Users/minghao/Desktop/personal/equity_lake/config/settings.yaml
Tickers YAML: /Users/minghao/Desktop/personal/equity_lake/config/tickers.yaml


## 1. Settings Overview

The `Settings` class uses Pydantic Settings with multiple sources. Let's load and inspect it.

In [2]:
settings = get_settings()

print("=== Project Settings ===")
print(f"  Name: {settings.project.name}")
print(f"  Version: {settings.project.version}")
print(f"  Environment: {settings.project.environment}")

print("\n=== Storage Settings ===")
print(f"  Data dir: {settings.storage.data_dir}")
print(f"  Lake dir: {settings.storage.lake_dir}")
print(f"  DB path: {settings.storage.db_path}")

print("\n=== Ingestion Settings ===")
print(f"  Default markets: {settings.ingestion.default_markets}")
print(f"  Parallel: {settings.ingestion.parallel}")
print(f"  Max workers: {settings.ingestion.max_workers}")
print(f"  Ticker config: {settings.ingestion.ticker_config_path}")

print("\n=== Schedule Settings ===")
print(f"  Cron: {settings.schedule.cron}")
print(f"  Timezone: {settings.schedule.timezone}")

=== Project Settings ===
  Name: equity-lake
  Version: 0.1.0
  Environment: development

=== Storage Settings ===
  Data dir: data
  Lake dir: data/lake
  DB path: equity_data.duckdb

=== Ingestion Settings ===
  Default markets: ['us', 'cn', 'hk_sg']
  Parallel: True
  Max workers: 3
  Ticker config: config/tickers.yaml

=== Schedule Settings ===
  Cron: 0 1 * * 1-5
  Timezone: Asia/Singapore


## 2. Settings as DataFrame

Convert settings to a flat table for easy inspection.

In [3]:
def flatten_settings(s: Settings) -> pd.DataFrame:
    rows = []
    for section_name in ["project", "storage", "ingestion", "schedule", "dashboard", "monitoring"]:
        section = getattr(s, section_name, None)
        if section is None:
            continue
        for field_name, value in section.model_dump().items():
            rows.append({
                "Section": section_name,
                "Field": field_name,
                "Value": str(value),
                "Type": type(section.__dict__[field_name]).__name__,
            })
    return pd.DataFrame(rows)

settings_df = flatten_settings(settings)
print(f"Total config fields: {len(settings_df)}")
settings_df

Total config fields: 23


,Section,Field,Value,Type
0,project,name,equity-lake,str
1,project,version,0.1.0,str
2,project,environment,development,str
3,storage,data_dir,data,str
4,storage,lake_dir,data/lake,str
5,storage,logs_dir,logs,str
6,storage,models_dir,data/models,str
7,storage,db_path,equity_data.duckdb,str
8,ingestion,default_markets,"['us', 'cn', 'hk_sg']",list
9,ingestion,parallel,True,bool


## 3. Ticker Universe

The `TickerConfig` loads all tickers from `tickers.yaml`. Let's explore the universe.

In [4]:
ticker_config = TickerConfig()

stats = ticker_config.get_stats()
print("=== Ticker Config Stats ===")
for key, value in stats.items():
    print(f"  {key}: {value}")

2026-06-09 09:25:54 [info     ] Loaded ticker config from /Users/minghao/Desktop/personal/equity_lake/config/tickers.yaml: 116 tickers across 6 markets


=== Ticker Config Stats ===
  version: 1.0
  total_markets: 6
  total_groups: 9
  markets: {'us': {'currency': 'USD', 'total_tickers': 50, 'active_tickers': 50, 'inactive_tickers': 0, 'exchanges': ['NYSE', 'NASDAQ'], 'sectors': ['Energy', 'Consumer Defensive', 'Communication Services', 'Consumer Cyclical', 'Industrials', 'Healthcare', 'Technology', 'Financial Services']}, 'cn': {'currency': 'CNY', 'total_tickers': 20, 'active_tickers': 20, 'inactive_tickers': 0, 'exchanges': ['SSE', 'SZSE'], 'sectors': ['Energy', 'Finance', 'Consumer staples', 'Consumer Cyclical', 'Healthcare', 'Technology']}, 'hk': {'currency': 'HKD', 'total_tickers': 16, 'active_tickers': 15, 'inactive_tickers': 1, 'exchanges': ['HKEX'], 'sectors': ['Energy', 'Communication Services', 'Finance', 'Real Estate', 'Consumer Cyclical', 'Industrials', 'Technology']}, 'sg': {'currency': 'SGD', 'total_tickers': 11, 'active_tickers': 11, 'inactive_tickers': 0, 'exchanges': ['SGX'], 'sectors': ['Communication Services', 'Finan

In [5]:
markets = ["us", "cn", "hk_sg", "jpx", "krx"]
all_tickers = []

for market in markets:
    try:
        tickers = ticker_config.list_tickers(market, include_metadata=True)
        for t in tickers:
            t["market"] = market
        all_tickers.extend(tickers)
        print(f"{market}: {len(tickers)} tickers")
    except Exception as e:
        print(f"{market}: {e}")

tickers_df = pd.DataFrame(all_tickers)
print(f"\nTotal tickers: {len(tickers_df)}")
tickers_df.head(10)

us: 'str' object does not support item assignment
cn: 'str' object does not support item assignment
hk_sg: 0 tickers
jpx: 'str' object does not support item assignment
krx: 'str' object does not support item assignment

Total tickers: 0


""


## 4. Ticker Treemap — Market → Sector

Visualize the ticker universe as a treemap grouped by market and sector.

In [6]:
if not tickers_df.empty and "sector" in tickers_df.columns:
    treemap_df = tickers_df.copy()
    treemap_df["sector"] = treemap_df["sector"].fillna("Unknown")

    fig = px.treemap(
        treemap_df,
        path=["market", "sector", "symbol"],
        values="priority",
        color="priority",
        color_continuous_scale="RdYlGn",
        title="Ticker Universe: Market → Sector → Symbol (colored by priority)",
    )
    fig.update_layout(height=600)
    fig.show()
else:
    print("Ticker metadata not available for treemap — check tickers.yaml format")

Ticker metadata not available for treemap — check tickers.yaml format


## 5. Priority Distribution

How are tickers distributed by priority level (1-10)?

In [7]:
if not tickers_df.empty and "priority" in tickers_df.columns:
    fig = px.histogram(
        tickers_df,
        x="priority",
        color="market",
        barmode="group",
        title="Ticker Priority Distribution by Market",
        labels={"priority": "Priority (1=lowest, 10=highest)", "count": "Number of Tickers"},
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("Priority data not available")

Priority data not available


## 6. Config Priority Demo

Demonstrate how config values cascade: init kwargs > env vars > .env > YAML.

In [8]:
import os

# Show current value from YAML
print(f"Default markets (from YAML/settings): {settings.ingestion.default_markets}")
print(f"Parallel (from YAML/settings): {settings.ingestion.parallel}")

# Override via env var (simulated)
os.environ["EQUITY_INGESTION__DEFAULT_MARKETS"] = '["us", "cn"]'
os.environ["EQUITY_INGESTION__PARALLEL"] = "false"

# Clear cache and reload
from equity_lake.core.config import clear_settings_cache
clear_settings_cache()

new_settings = get_settings()
print(f"\nDefault markets (after env override): {new_settings.ingestion.default_markets}")
print(f"Parallel (after env override): {new_settings.ingestion.parallel}")

# Clean up
del os.environ["EQUITY_INGESTION__DEFAULT_MARKETS"]
del os.environ["EQUITY_INGESTION__PARALLEL"]
clear_settings_cache()

Default markets (from YAML/settings): ['us', 'cn', 'hk_sg']
Parallel (from YAML/settings): True

Default markets (after env override): ['us', 'cn']
Parallel (after env override): False


## 7. Ticker Filtering

The `TickerConfig` provides rich filtering via tags, sectors, groups, and priority thresholds.

In [9]:
# Filter by tag
try:
    tech_tickers = ticker_config.get_tickers_by_tag("tech", market="us")
    print(f"US tech tickers: {tech_tickers}")
except Exception as e:
    print(f"Tag filter: {e}")

# Filter by minimum priority
try:
    high_priority = ticker_config.get_tickers_for_market("us", min_priority=7)
    print(f"\nUS high-priority (>=7): {high_priority}")
except Exception as e:
    print(f"Priority filter: {e}")

# Filter by group
try:
    groups = ticker_config.get_groups()
    print(f"\nAvailable groups: {groups}")
    for g in groups[:3]:
        info = ticker_config.get_group_info(g)
        if info:
            print(f"  {g}: {info}")
except Exception as e:
    print(f"Group filter: {e}")

US tech tickers: []

US high-priority (>=7): ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'BRK-B', 'TSLA', 'JPM', 'V', 'MA', 'UNH', 'JNJ', 'LLY', 'AVGO', 'WMT', 'PG', 'XOM', 'CVX', 'BAC', 'WFC', 'TMO', 'MRK', 'ABT', 'KO', 'PEP', 'COST', 'HD', 'MCD', 'NKE', 'DIS', 'NFLX', 'COP', 'CAT', 'BA', 'GE', 'HON', 'UNP', 'ADBE', 'CRM', 'ORCL', 'AMD', 'QCOM', 'VZ', 'DHR', 'LIN', 'INTC', 'CSCO', 'IBM', 'CMCSA']

Available groups: ['faang', 'faang_plus', 'sp500_top_10', 'dividend_aristocrats', 'asian_banks', 'chinese_tech', 'us_semiconductors', 'us_healthcare', 'us_banking']
  faang: description='Big 5 tech companies (FAANG)' markets=['us'] tickers=['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META']
  faang_plus: description='FAANG + NVDA + TSLA' markets=['us'] tickers=['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META', 'NVDA', 'TSLA']
  sp500_top_10: description='Top 10 S&P 500 by market cap' markets=['us'] tickers=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'LLY', 'JPM', 'V']


## 8. Config Validation

Validate YAML configs for consistency and completeness.

In [10]:
try:
    is_valid = ticker_config.validate_config()
    print(f"Config validation: {'PASSED' if is_valid else 'FAILED'}")
except Exception as e:
    print(f"Validation result: {e}")

# Check watchlist
from pathlib import Path
watchlist_path = CONFIG_DIR / "watchlist.yaml"
if watchlist_path.exists():
    print(f"\nWatchlist file exists: {watchlist_path}")
else:
    print("\nNo watchlist.yaml found")

Config validation: PASSED

Watchlist file exists: /Users/minghao/Desktop/personal/equity_lake/config/watchlist.yaml


## Next Steps

- **[02-data-ingestion.ipynb](02-data-ingestion.ipynb)** — Ingest market data using configured tickers
- **[03-storage-and-queries.ipynb](03-storage-and-queries.ipynb)** — Query ingested data via DuckDB
- **[04-hamilton-features.ipynb](04-hamilton-features.ipynb)** — Generate features using the Hamilton DAG